# BMC Discovery OS Health Check

Collect operating-system and appliance-service health evidence directly over SSH. This notebook deliberately does not use the Discovery REST API.

## Goal

Run the standard non-destructive SSH checks for one or more appliances configured in `config.yaml`: appliance health check, host identity, disk capacity, time synchronisation, DNS, service status, cluster state, Discovery service logs, core dumps, cron jobs, and playback data. Each check is recorded independently so a failure does not hide the remaining evidence.

## Setup

The notebook reads `target`, `username`, and either `password`, `password_file`, or legacy `f_passwd` from `config.yaml`. Keep SSH secrets in that local configuration or a password file; do not place them in this notebook. Relative password-file paths are resolved from the repository root.

In [ ]:
from pathlib import Path
from typing import Any, Dict, List, Optional
import sys

try:
    from IPython.display import HTML, display
except ImportError:
    class HTML(str):
        pass

    def display(value):
        print(value)


def find_repo_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / 'config.yaml').exists() or (candidate / '.git').is_dir():
            return candidate
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import tideway
from tideway import notebooks as tw_nb
from tideway.results import ReportResult, TextResult

pd.set_option('display.max_colwidth', None)
display(HTML('''
<style>
.jp-Notebook .jp-OutputArea-output table, .jp-Notebook table.dataframe,
.jp-RenderedHTMLCommon table, table.dataframe { width: 100% !important; }
.jp-Notebook table.dataframe th, .jp-Notebook table.dataframe td,
table.dataframe th, table.dataframe td { white-space: normal !important; word-break: break-word; }
</style>
'''))


def show_table(frame: pd.DataFrame) -> None:
    if frame is None or frame.empty:
        print('No rows to display.')
        return
    html = frame.to_html(border=1, index=False)
    html = html.replace('<table border="1" class="dataframe">', '<table border="1" class="dataframe" style="width:100%">', 1)
    display(HTML(html))


## Controls

Use `INCLUDE_APPLIANCES` to select configured appliance names or targets. Set `WRITE_OUTPUT` to save the raw SSH evidence plus a summary CSV in the canonical `output_<target>` directory.

In [ ]:
INCLUDE_APPLIANCES = None  # for example: ['sandbox']
EXCLUDE_APPLIANCES = []

WRITE_OUTPUT = True
OUTPUT_BASE_DIR = None
DISK_USED_THRESHOLD = 70
TEXT_PREVIEW_LINES = 25

# Optional non-secret overrides. Leave these as None to use config.yaml.
SSH_USERNAME = None
SSH_PASSWORD_FILE = None
SYSTEM_USERNAME = None
SYSTEM_PASSWORD_FILE = None

SSH_CHECKS = [
    'health_check',
    'host_info',
    'disk_info',
    'disk_usage_alerts',
    'ntp',
    'dns_resolution',
    'syslog',
    'vmware_tools',
    'core_dumps',
    'clustering',
    'ds_status',
    'cmdb_errors',
    'tw_crontab',
    'playback_data',
]


## Load appliance configuration

The inventory below confirms which SSH targets are ready without displaying passwords or their contents.

In [ ]:
def appliance_entries(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    entries = config.get('appliances') or []
    if isinstance(entries, dict):
        entries = [entries]
    return entries or [{'name': config.get('name') or config.get('target') or 'default'}]


def resolve_optional_path(value: Optional[str], repo_root: Path) -> Optional[str]:
    if not value:
        return None
    path = Path(value).expanduser()
    return str(path if path.is_absolute() else repo_root / path)


config = tw_nb.load_config()
include_set = set(INCLUDE_APPLIANCES or [])
exclude_set = set(EXCLUDE_APPLIANCES or [])
appliances = []
configuration_issues = []

for raw_entry in appliance_entries(config):
    entry = dict(config)
    entry.update(raw_entry or {})
    entry.pop('appliances', None)
    target = str(entry.get('target') or '').strip()
    name = str(entry.get('name') or target or 'default').strip()
    if include_set and name not in include_set and target not in include_set:
        continue
    if name in exclude_set or target in exclude_set:
        continue
    if not target:
        configuration_issues.append({'Appliance': name, 'Issue': 'Missing target in config.yaml'})
        continue

    password_file = resolve_optional_path(SSH_PASSWORD_FILE or entry.get('password_file') or entry.get('f_passwd'), REPO_ROOT)
    system_password_file = resolve_optional_path(SYSTEM_PASSWORD_FILE or entry.get('system_password_file'), REPO_ROOT)
    cli_kwargs = {
        'username': SSH_USERNAME or entry.get('username') or 'tideway',
        'password': entry.get('password'),
        'password_file': password_file,
        'system_username': SYSTEM_USERNAME or entry.get('system_username') or 'system',
        'system_password': entry.get('system_password'),
        'system_password_file': system_password_file,
    }
    appliances.append({
        'name': name,
        'target': target,
        'cli_kwargs': cli_kwargs,
        'output_dir': tw_nb.output_dir_for(target, OUTPUT_BASE_DIR),
        'ssh_ready': bool(cli_kwargs['password'] or password_file),
    })

inventory = pd.DataFrame([
    {
        'Appliance': item['name'],
        'Target': item['target'],
        'SSH User': item['cli_kwargs']['username'],
        'SSH Ready': item['ssh_ready'],
        'Output Directory': str(item['output_dir']),
    }
    for item in appliances
])
show_table(inventory)
show_table(pd.DataFrame(configuration_issues))


## Run SSH health checks

All commands are read-only diagnostic commands. The notebook does not call destructive maintenance operations such as queue clearing.

In [ ]:
def result_row_count(result: Any) -> Optional[int]:
    if isinstance(result, ReportResult):
        return len(result.rows)
    if isinstance(result, TextResult):
        return len(result.text.splitlines())
    return None


def run_check(cli, check_name: str, output_dir: Optional[str]):
    kwargs = {'output_dir': output_dir} if output_dir else {}
    if check_name == 'disk_usage_alerts':
        return cli.disk_usage_alerts(threshold=DISK_USED_THRESHOLD, **kwargs)
    return getattr(cli, check_name)(**kwargs)


results: Dict[tuple, Any] = {}
status_rows: List[Dict[str, Any]] = []

for item in appliances:
    if not item['ssh_ready']:
        for check_name in SSH_CHECKS:
            status_rows.append({
                'Appliance': item['name'], 'Check': check_name, 'Status': 'skipped',
                'Rows': None, 'Evidence files': '',
                'Detail': 'Missing SSH password, password_file, or f_passwd in config.yaml',
            })
        continue

    output_dir = str(item['output_dir']) if WRITE_OUTPUT else None
    try:
        with tideway.appliance_cli(item['target'], **item['cli_kwargs']) as cli:
            for check_name in SSH_CHECKS:
                try:
                    result = run_check(cli, check_name, output_dir)
                    results[(item['name'], check_name)] = result
                    status_rows.append({
                        'Appliance': item['name'], 'Check': check_name, 'Status': 'ok',
                        'Rows': result_row_count(result),
                        'Evidence files': ', '.join(getattr(result, 'files', [])), 'Detail': '',
                    })
                except Exception as exc:
                    status_rows.append({
                        'Appliance': item['name'], 'Check': check_name, 'Status': 'error',
                        'Rows': None, 'Evidence files': '', 'Detail': str(exc),
                    })
    except Exception as exc:
        for check_name in SSH_CHECKS:
            if (item['name'], check_name) not in results:
                status_rows.append({
                    'Appliance': item['name'], 'Check': check_name, 'Status': 'error',
                    'Rows': None, 'Evidence files': '', 'Detail': f'SSH connection failed: {exc}',
                })

summary = pd.DataFrame(status_rows)
show_table(summary)


## Review the collected evidence

Text results are previewed below; CSV-style checks are shown as tables. Full raw evidence is retained in the output directory when enabled.

In [ ]:
for (appliance_name, check_name), result in results.items():
    print(f'## {appliance_name}: {check_name}')
    if isinstance(result, ReportResult):
        show_table(pd.DataFrame(result.rows, columns=result.headers))
    elif isinstance(result, TextResult):
        preview = '\n'.join(result.text.splitlines()[:TEXT_PREVIEW_LINES])
        print(preview or '(no output)')
    else:
        print(result)

if WRITE_OUTPUT and not summary.empty:
    for item in appliances:
        appliance_summary = summary[summary['Appliance'] == item['name']]
        if not appliance_summary.empty:
            item['output_dir'].mkdir(parents=True, exist_ok=True)
            appliance_summary.to_csv(item['output_dir'] / 'os_healthcheck_summary.csv', index=False)
            print(f"Saved summary: {item['output_dir'] / 'os_healthcheck_summary.csv'}")


## Next steps

Investigate every `error` entry in the summary before treating the appliance as healthy. Review disk rows over the configured threshold, service failures, core dumps, and errors in the Discovery service logs. Re-run only a selected appliance with `INCLUDE_APPLIANCES` after making a remediation.